# Deep-Dive E-commerce Data Analytics: TeddyWorld Case Study

Sổ tay này thực hiện phân tích chuyên sâu dữ liệu thương mại điện tử theo quy trình chuẩn, từ cái nhìn tổng quan đến các ngách tối ưu lợi nhuận cụ thể.

## Part I: Setup & Data Ingestion (Khởi tạo và Tải dữ liệu)

In [ ]:
# 1. Import các thư viện cần thiết
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Thiết lập phong cách biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# 2. Xác định đường dẫn và tải dữ liệu
data_path = 'dataset/' if os.path.exists('dataset/') else ''

orders = pd.read_csv(data_path + 'orders.csv')
sessions = pd.read_csv(data_path + 'website_sessions.csv')
products = pd.read_csv(data_path + 'products.csv')
order_items = pd.read_csv(data_path + 'order_items.csv')
refunds = pd.read_csv(data_path + 'order_item_refunds.csv')
pageviews = pd.read_csv(data_path + 'website_pageviews.csv')

print("✅ Đã tải dữ liệu thành công!")

## Part II: Data Cleaning & Pre-processing (Tiền xử lý dữ liệu)

In [ ]:
# 1. Chuyển đổi định dạng thời gian và tạo cột 'month' cho tất cả bảng
for df in [orders, sessions, refunds, order_items, pageviews]:
    df['created_at'] = pd.to_datetime(df['created_at'])
    df['month'] = df['created_at'].dt.to_period('M').astype(str)

# 2. Xử lý giá trị thiếu dựa trên thực tế kinh doanh
sessions['utm_source'] = sessions['utm_source'].fillna('organic')
sessions['utm_campaign'] = sessions['utm_campaign'].fillna('none')
sessions['utm_content'] = sessions['utm_content'].fillna('none')
sessions['http_referer'] = sessions['http_referer'].fillna('direct_traffic')

# 3. Chuẩn hóa dữ liệu văn bản
sessions['device_type'] = sessions['device_type'].str.lower()

print("✅ Đã hoàn thành làm sạch và tạo cột thời gian!")

## Part III: General Overview Metrics (Chỉ số Kinh doanh Tổng quan)

In [ ]:
# Tính toán các chỉ số quan trọng
total_sessions = sessions['website_session_id'].nunique()
total_orders = orders['order_id'].nunique()
total_revenue = orders['price_usd'].sum()
total_refunds = refunds['refund_amount_usd'].sum()
total_cogs = orders['cogs_usd'].sum()

overall_cr = (total_orders / total_sessions) * 100
overall_net_profit = total_revenue - total_refunds - total_cogs
overall_net_margin = (overall_net_profit / total_revenue) * 100
avg_order_value = total_revenue / total_orders

summary_metrics = pd.DataFrame({
    'Chỉ số': ['Sessions', 'Orders', 'CR (%)', 'Revenue', 'Refunds', 'Net Profit', 'Net Margin (%)', 'AOV'],
    'Giá trị': [
        f"{total_sessions:,}", f"{total_orders:,}", f"{overall_cr:.2f}%", 
        f"${total_revenue:,.2f}", f"${total_refunds:,.2f}", f"${overall_net_profit:,.2f}", 
        f"{overall_net_margin:.2f}%", f"${avg_order_value:,.2f}"
    ]
})

print("--- BẢNG CHỈ SỐ KINH DOANH TỔNG QUAN ---")
display(summary_metrics)

## Part IV: Unit Economics (Biz Unit - Lợi nhuận theo Sản phẩm)

In [ ]:
# 1. Xử lý và Tổng hợp dữ liệu lợi nhuận
items_full = order_items.merge(products, on='product_id', how='left')
items_full = items_full.merge(refunds, on='order_item_id', how='left')
items_full['refund_amount_usd'] = items_full['refund_amount_usd'].fillna(0)
items_full['gross_profit'] = items_full['price_usd'] - items_full['refund_amount_usd'] - items_full['cogs_usd']

unit_economics = items_full.groupby('product_name').agg(
    revenue=('price_usd', 'sum'),
    net_profit=('gross_profit', 'sum')
).reset_index()
unit_economics['margin_pct'] = (unit_economics['net_profit'] / unit_economics['revenue']) * 100
unit_economics = unit_economics.sort_values('margin_pct', ascending=False)

print("--- BẢNG LỢI NHUẬN BIÊN (NET MARGIN %) THEO SẢN PHẨM ---")
display(unit_economics)

In [ ]:
# 2. Vẽ biểu đồ Net Margin %
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=unit_economics, x='margin_pct', y='product_name', palette='magma')
for p in ax.patches:
    ax.annotate(f'{p.get_width():.1f}%', (p.get_width(), p.get_y() + p.get_height() / 2.), ha='left', va='center', fontweight='bold')
plt.title('Tỷ suất lợi nhuận thực tế theo Sản phẩm', fontweight='bold')
plt.show()

> **INSIGHT**: `The Hudson River Mini bear` có margin cao nhất (**67.1%**), trong khi `The Original Mr. Fuzzy` thấp nhất (**55.9%**). Cần đẩy mạnh các dòng bear có margin > 65%.

## Part V: Monthly Revenue Trends (Xu hướng theo Thời gian)

In [ ]:
# Chuẩn bị dữ liệu thời gian
order_items['month_year'] = order_items['created_at'].dt.to_period('M').astype(str)
ts_data = order_items.merge(products, on='product_id', how='left').groupby(['month_year', 'product_name'])['price_usd'].sum().reset_index()
pivot_ts = ts_data.pivot(index='month_year', columns='product_name', values='price_usd').fillna(0)

print("--- DOANH THU THEO THÁNG VÀ SẢN PHẨM ---")
display(pivot_ts)

In [ ]:
# Vẽ biểu đồ xu hướng
plt.figure(figsize=(14, 7))
sns.lineplot(data=ts_data, x='month_year', y='price_usd', hue='product_name', marker='o', linewidth=2.5)
plt.title('Xu hướng Doanh thu theo Tháng', fontweight='bold')
plt.xticks(rotation=45)
plt.show()

> **INSIGHT**: Tăng trưởng tích lũy đạt **2,532%** sau 3 năm. Doanh thu bùng nổ mạnh nhất vào các mùa lễ hội cuối năm.

## Part VI: Traffic & Conversion Analysis (Lưu lượng & Chuyển đổi)

### 1. Phân phối Traffic (Source & Device Overview)

In [ ]:
# Tính toán phân phối traffic
source_dist = sessions['utm_source'].value_counts(normalize=True).mul(100).reset_index()
device_dist = sessions['device_type'].value_counts(normalize=True).mul(100).reset_index()

print("--- PHÂN PHỐI NGUỒN TRAFFIC (%) ---")
display(source_dist)
print("\n--- PHÂN PHỐI THIẾT BỊ (%) ---")
display(device_dist)

In [ ]:
# Vẽ biểu đồ phân phối
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=source_dist, x='utm_source', y='proportion', ax=ax1, palette='coolwarm')
ax1.set_title('Nguồn Traffic (%)', fontweight='bold')
ax2.pie(device_dist['proportion'], labels=device_dist['device_type'], autopct='%1.1f%%', startangle=140, colors=['#66b3ff','#99ff99'])
ax2.set_title('Thiết bị (%)', fontweight='bold')
plt.show()

> **NHẬN XÉT (INSIGHT):**
> - **Sự thống trị của Google Search**: `gsearch` là nguồn traffic lớn nhất, chiếm tới **66.8%** tổng lượng truy cập. Điều này cho thấy website phụ thuộc rất lớn vào tìm kiếm trả phí/SEO từ Google.
> - **Ưu thế tuyệt đối của Desktop**: Gần **70%** (69.2%) người dùng truy cập từ máy tính để bàn. Đây là tệp khách hàng chính cần tập trung tối ưu hóa trải nghiệm.
> - **Organic Traffic ổn định**: Xếp thứ hai with **17.6%**, cho thấy thương hiệu đã có một lượng khách hàng trung thành tự tìm kiếm và truy cập trực tiếp.
> - **Tiềm năng chưa khai phá**: Các kênh Social (**2.3%**) còn rất khiêm tốn, mở ra cơ hội tăng trưởng lớn nếu đầu tư vào Marketing mạng xã hội.

### 2. Tỷ lệ chuyển đổi chi tiết theo Thiết bị & Nguồn

In [ ]:
# Tính CR theo Device & Source
session_orders = sessions.merge(orders, on='website_session_id', how='left')
source_device_perf = session_orders.groupby(['device_type', 'utm_source']).agg(sessions=('website_session_id', 'count'), orders=('order_id', 'count')).reset_index()
source_device_perf['cr_pct'] = (source_device_perf['orders'] / source_device_perf['sessions']) * 100

print("--- BẢNG HIỆU QUẢ CHUYỂN ĐỔI CHI TIẾT ---")
display(source_device_perf.sort_values('cr_pct', ascending=False))

In [ ]:
# Vẽ biểu đồ CR đa chiều
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=source_device_perf, x='utm_source', y='cr_pct', hue='device_type', palette='viridis')
for p in ax.patches:
    if p.get_height() > 0: ax.annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontweight='bold')
plt.title('CR% theo Thiết bị và Nguồn', fontweight='bold')
plt.show()

> **NHẬN XÉT (INSIGHT):**
> - **Organic Desktop là "Vua" về hiệu quả**: Mặc dù chỉ đứng thứ 2 về lượng traffic (**17.6%**), nhưng nguồn **Organic** trên **Desktop** lại có tỷ lệ chuyển đổi cao nhất sàn (**9.89%**). Đây là nhóm khách hàng chất lượng nhất.
> - **Khoảng cách CR cực lớn (Efficiency Gap)**: Tỷ lệ chuyển đổi trên Desktop cao gấp **~3 lần** so với Mobile trên hầu hết các nguồn. Ví dụ, `gsearch` trên Desktop đạt **8.43%** nhưng trên Mobile chỉ là **3.17%**.
> - **Vấn đề tối ưu hóa di động**: Tất cả các nguồn traffic trên Mobile đều có CR rất thấp (dưới **3.2%**). Đặc biệt, nguồn **Socialbook** trên Mobile gần như không hiệu quả với CR chỉ **0.83%**.
> - **Chiến lược**: Cần ưu tiên ngân sách cho Google Search Desktop và tối ưu hóa SEO để duy trì traffic Organic. Đồng thời, cần một cuộc cải tổ lớn về trải nghiệm Mobile để không lãng phí 30.8% lượng traffic hiện có.

### 3. Phân tích Phễu chuyển đổi tương tác (Interactive Side-by-Side Funnel)

In [ ]:
# 1. Chuẩn bị dữ liệu Phễu cho Plotly
pv_device = pageviews.merge(sessions[['website_session_id', 'device_type']], on='website_session_id', how='left')

def get_funnel_df(df, device_name):
    steps = [
        {'Step': '1. Landed', 'Sessions': df['website_session_id'].nunique()},
        {'Step': '2. Product', 'Sessions': df[df['pageview_url'].str.contains('/products|/the-original|/the-forever|/the-birthday|/the-hudson')]['website_session_id'].nunique()},
        {'Step': '3. Cart', 'Sessions': df[df['pageview_url'] == '/cart']['website_session_id'].nunique()},
        {'Step': '4. Shipping', 'Sessions': df[df['pageview_url'] == '/shipping']['website_session_id'].nunique()},
        {'Step': '5. Billing', 'Sessions': df[df['pageview_url'].str.contains('/billing')]['website_session_id'].nunique()},
        {'Step': '6. Thank You', 'Sessions': df[df['pageview_url'] == '/thank-you-for-your-order']['website_session_id'].nunique()}
    ]
    res = pd.DataFrame(steps)
    res['Device'] = device_name
    return res

funnel_desktop = get_funnel_df(pv_device[pv_device['device_type'] == 'desktop'], 'Desktop')
funnel_mobile = get_funnel_df(pv_device[pv_device['device_type'] == 'mobile'], 'Mobile')

print("--- DỮ LIỆU PHỄU TÓM TẮT --- ")
display(funnel_desktop, funnel_mobile)

In [ ]:
# 2. Vẽ biểu đồ Funnel Subplots bằng Plotly
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Funnel: DESKTOP", "Funnel: MOBILE"),
                    specs=[[{"type": "funnel"}, {"type": "funnel"}]])

fig.add_trace(go.Funnel(
    name = 'Desktop',
    y = funnel_desktop['Step'],
    x = funnel_desktop['Sessions'],
    textinfo = "value+percent initial+percent previous",
    marker = {"color": "#1f77b4"}
), row=1, col=1)

fig.add_trace(go.Funnel(
    name = 'Mobile',
    y = funnel_mobile['Step'],
    x = funnel_mobile['Sessions'],
    textinfo = "value+percent initial+percent previous",
    marker = {"color": "#d62728"}
), row=1, col=2)

fig.update_layout(
    title_text='Interactive Side-by-Side Funnel: Desktop vs Mobile',
    height=600,
    showlegend=False
)

fig.show()

> **INSIGHT**: Bố cục side-by-side giúp ta so sánh trực tiếp độ dốc của phễu. Phễu Mobile dốc hơn đáng kể ở ngay bước đầu tiên, minh chứng cho sự sụt giảm hiệu quả trải nghiệm di động.

### 4. Phân tích Tỷ lệ Drop-off chi tiết theo Thiết bị (Device Drop-off Comparison)

So sánh tỷ lệ rời bỏ phễu giữa Desktop và Mobile tại các điểm gãy quan trọng: Product -> Cart và Billing -> Thank You.

In [ ]:
# 1. Chuẩn bị dữ liệu Drop-off
pv_full = pageviews.merge(sessions[['website_session_id', 'device_type']], on='website_session_id', how='left')

# Tính toán Overall Drop-off
def get_overall_dropoff_stats(df):
    prod = df[df['pageview_url'].str.contains('/products|/the-original|/the-forever|/the-birthday|/the-hudson')]['website_session_id'].nunique()
    cart = df[df['pageview_url'] == '/cart']['website_session_id'].nunique()
    billing = df[df['pageview_url'].str.contains('/billing')]['website_session_id'].nunique()
    order = df[df['pageview_url'] == '/thank-you-for-your-order']['website_session_id'].nunique()
    return prod, cart, billing, order

d_prod, d_cart, d_bill, d_order = get_overall_dropoff_stats(pv_full[pv_full['device_type'] == 'desktop'])
m_prod, m_cart, m_bill, m_order = get_overall_dropoff_stats(pv_full[pv_full['device_type'] == 'mobile'])

summary_dropoff = pd.DataFrame({
    'Giai đoạn': ['Product -> Cart', 'Billing -> Thank You'],
    'Desktop (%)': [(1 - d_cart / d_prod) * 100, (1 - d_order / d_bill) * 100],
    'Mobile (%)': [(1 - m_cart / m_prod) * 100, (1 - m_order / m_bill) * 100]
})

# 2. Tính toán Drop-off theo thời gian
def get_monthly_dropoff(df):
    product = df[df['pageview_url'].str.contains('/products|/the-original|/the-forever|/the-birthday|/the-hudson')].groupby('month')['website_session_id'].nunique()
    cart = df[df['pageview_url'] == '/cart'].groupby('month')['website_session_id'].nunique()
    billing = df[df['pageview_url'].str.contains('/billing')].groupby('month')['website_session_id'].nunique()
    order = df[df['pageview_url'] == '/thank-you-for-your-order'].groupby('month')['website_session_id'].nunique()
    res = pd.DataFrame({'Product': product, 'Cart': cart, 'Billing': billing, 'Order': order}).fillna(0)
    res['Dropoff_Prod_to_Cart'] = (1 - res['Cart'] / res['Product']) * 100
    res['Dropoff_Billing_to_Order'] = (1 - res['Order'] / res['Billing']) * 100
    return res

dropoff_desktop_ts = get_monthly_dropoff(pv_full[pv_full['device_type'] == 'desktop'])
dropoff_mobile_ts = get_monthly_dropoff(pv_full[pv_full['device_type'] == 'mobile'])

print("--- TỶ LỆ DROP-OFF TỔNG THỂ (%) ---")
display(summary_dropoff)

In [ ]:
# 2. Vẽ biểu đồ so sánh xu hướng Drop-off theo thiết bị (Time-series)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Subplot 1: Product to Cart
sns.lineplot(data=dropoff_desktop_ts, x=dropoff_desktop_ts.index, y='Dropoff_Prod_to_Cart', marker='o', label='Desktop', ax=ax1, linewidth=2)
sns.lineplot(data=dropoff_mobile_ts, x=dropoff_mobile_ts.index, y='Dropoff_Prod_to_Cart', marker='s', label='Mobile', ax=ax1, linewidth=2, color='red')
ax1.set_title('Drop-off: Product -> Cart (%)', fontweight='bold')
ax1.set_ylabel('Drop-off Rate (%)')
ax1.set_ylim(0, 80)
ax1.tick_params(axis='x', rotation=45)

# Subplot 2: Billing to Thank You
sns.lineplot(data=dropoff_desktop_ts, x=dropoff_desktop_ts.index, y='Dropoff_Billing_to_Order', marker='o', label='Desktop', ax=ax2, linewidth=2)
sns.lineplot(data=dropoff_mobile_ts, x=dropoff_mobile_ts.index, y='Dropoff_Billing_to_Order', marker='s', label='Mobile', ax=ax2, linewidth=2, color='red')
ax2.set_title('Drop-off: Billing -> Thank You (%)', fontweight='bold')
ax2.set_ylabel('Drop-off Rate (%)')
ax2.set_ylim(0, 80)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

> **NHẬN XÉT CHI TIẾT TỪ DỮ LIỆU XU HƯỚNG (DETAILED TREND INSIGHTS):**
> 
> - **Hiệu suất Desktop cải thiện vượt trội**: Dữ liệu từ `dropoff_desktop_ts` cho thấy Desktop giảm drop-off mạnh ở bước Thanh toán (**Billing -> Order**) từ **51.92% (năm 2012)** xuống chỉ còn **33.55% (năm 2015)** sau 3 năm tối ưu hóa. Đây là minh chứng cho việc cải thiện UI/UX trên máy tính đang đi đúng hướng.
> - **Mobile tụt hậu và không ổn định**: Trái ngược với Desktop, Mobile luôn có tỷ lệ drop-off cao hơn đáng kể, chênh lệch từ **7% đến 10%** ngay tại bước đầu tiên (**Product -> Cart**). Tốc độ cải thiện của Mobile rất chậm và dữ liệu cho thấy sự biến động mạnh (volatile) qua từng tháng.
> - **Khoảng cách lớn nhất (The Conversion Gap)**: Tỷ lệ drop-off ở bước Thanh toán (**Billing -> Order**) trên Mobile có sự chênh lệch lớn nhất so với Desktop, đỉnh điểm lên đến **14% vào năm 2015**. Điều này cho thấy trong khi người dùng máy tính ngày càng dễ dàng chốt đơn, thì người dùng điện thoại vẫn đang gặp phải những rào cản thanh toán cực kỳ lớn.

## Part VII: AOV & Cross-selling Optimization (Tối ưu Bán chéo)

### 1. Phân tích AOV & Cross-selling

In [ ]:
# Tính toán AOV
orders['order_type'] = orders['items_purchased'].apply(lambda x: 'Single' if x == 1 else 'Cross-sell')
aov_data = orders.groupby('order_type').agg(orders=('order_id', 'count'), revenue=('price_usd', 'sum')).reset_index()
aov_data['aov'] = aov_data['revenue'] / aov_data['orders']

print("--- SO SÁNH AOV GIỮA ĐƠN HÀNG LẺ VÀ BÁN KÈM ---")
display(aov_data)

In [ ]:
# Vẽ biểu đồ AOV
plt.figure(figsize=(8, 5))
ax = sns.barplot(data=aov_data, x='order_type', y='aov', palette='Set2')
for p in ax.patches:
    ax.annotate(f'${p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontweight='bold')
plt.title('So sánh AOV ($)', fontweight='bold')
plt.show()

### 2. Market Basket Analysis (MBA) - Combo Identification

Tính toán các chỉ số Support, Confidence và Lift cho các cặp sản phẩm.

In [ ]:
# 1. Xử lý dữ liệu MBA
order_items_named = order_items.merge(products[['product_id', 'product_name']], on='product_id', how='left')
df_mba = order_items_named.merge(order_items_named, on='order_id', suffixes=('_A', '_B'))
df_mba = df_mba[df_mba['product_id_A'] < df_mba['product_id_B']]

df_freq = df_mba.groupby(['product_name_A', 'product_name_B'], as_index=False).agg(total_orders_A_B=('order_id', 'nunique'))
total_orders_global = order_items['order_id'].nunique()
df_freq['support_A_B'] = df_freq['total_orders_A_B'] / total_orders_global

df_freq_A = order_items_named.groupby('product_name').agg(total_orders_A=('order_id', 'nunique')).reset_index().rename(columns={'product_name': 'product_name_A'})
df_freq_B = order_items_named.groupby('product_name').agg(total_orders_B=('order_id', 'nunique')).reset_index().rename(columns={'product_name': 'product_name_B'})

df_freq_final = df_freq.merge(df_freq_A, on='product_name_A', how='inner')
df_freq_final = df_freq_final.merge(df_freq_B, on='product_name_B', how='inner')
df_freq_final['confidence_A_B'] = df_freq_final['total_orders_A_B'] / df_freq_final['total_orders_A']
df_freq_final['lift_A_B'] = df_freq_final['confidence_A_B'] / (df_freq_final['total_orders_B'] / total_orders_global)
df_freq_final['combo_name'] = df_freq_final['product_name_A'] + " + " + df_freq_final['product_name_B']

print("✅ Đã xử lý xong dữ liệu Combo!")

#### 2.1 Trực quan hóa Top Combo theo LIFT (Sức mạnh cộng hưởng)

In [ ]:
# Top 10 Combo theo Lift
top_lift = df_freq_final.sort_values('lift_A_B', ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(data=top_lift, x='lift_A_B', y='combo_name', palette='Blues_r')
plt.title('Top 10 Combo theo LIFT (Sức mạnh cộng hưởng)', fontweight='bold')
plt.xlabel('Lift Value')
plt.show()

#### 2.2 Trực quan hóa Top Combo theo CONFIDENCE (Tỷ lệ mua kèm)

In [ ]:
# Top 10 Combo theo Confidence
top_conf = df_freq_final.sort_values('confidence_A_B', ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(data=top_conf, x='confidence_A_B', y='combo_name', palette='Greens_r')
plt.title('Top 10 Combo theo CONFIDENCE (Tỷ lệ mua kèm)', fontweight='bold')
plt.xlabel('Confidence Value')
plt.show()

> **NHẬN XÉT CHI CHIẾT (STRATEGIC INSIGHTS):**
> - **Dựa trên Lift**: Các combo này có chỉ số Lift cao nhất, thể hiện mối quan hệ "mua sản phẩm này thường sẽ mua sản phẩm kia". Đây là các cặp bài trùng để chạy quảng cáo Bundle.
> - **Dựa trên Confidence**: Các combo này có tỷ lệ mua kèm cao nhất. Dù Lift có thể không cao (do sản phẩm B quá phổ biến), nhưng đây là những lựa chọn an toàn nhất để hiển thị ở mục "Sản phẩm gợi ý" tại bước thanh toán.